In [81]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import scipy.io as io
import random
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import dense_to_sparse
from torch_geometric.data import Data, Batch
import torch.optim as optim
import torch.nn.functional as F

In [108]:
interictal_1 = '//Users/poorvaichandrasen/Downloads/Patient_1/Patient_1_interictal_segment_0001.mat'
i_eg_1 = io.loadmat(interictal_1)
interictal_segment_1 = i_eg_1.get('interictal_segment_1')
preictal_1 = '//Users/poorvaichandrasen/Downloads/Patient_1/Patient_1_preictal_segment_0001.mat'
p_eg_1 = io.loadmat(preictal_1)
preictal_segment_1 = p_eg_1.get('preictal_segment_1')
print(i_eg_1)

{'__header__': b'MATLAB 5.0 MAT-file, Platform: MACI64, Created on: Thu Aug 21 01:00:00 2014', '__version__': '1.0', '__globals__': [], 'interictal_segment_1': array([[(array([[ 122,  122,  122, ...,   34,   33,   31],
               [ 641,  642,  642, ..., -385, -387, -389],
               [ 358,  358,  356, ...,  -36,  -37,  -40],
               ...,
               [ 379,  380,  379, ...,  142,  141,  140],
               [1425, 1423, 1422, ..., -809, -809, -810],
               [ 589,  590,  591, ..., -489, -490, -493]], dtype=int16), array([[600]], dtype=uint16), array([[5000]], dtype=uint16), array([[array(['LD_1'], dtype='<U4'), array(['LD_3'], dtype='<U4'),
                array(['LD_4'], dtype='<U4'), array(['LD_5'], dtype='<U4'),
                array(['LD_6'], dtype='<U4'), array(['LD_7'], dtype='<U4'),
                array(['LD_8'], dtype='<U4'), array(['RD_1'], dtype='<U4'),
                array(['RD_2'], dtype='<U4'), array(['RD_3'], dtype='<U4'),
                array([

In [231]:
i_data= interictal_segment_1[0][0][0]
p_data = preictal_segment_1[0][0][0]
print(i_data)
print('_'*100)
print(i_data.shape)

[[ 122  122  122 ...   34   33   31]
 [ 641  642  642 ... -385 -387 -389]
 [ 358  358  356 ...  -36  -37  -40]
 ...
 [ 379  380  379 ...  142  141  140]
 [1425 1423 1422 ... -809 -809 -810]
 [ 589  590  591 ... -489 -490 -493]]
____________________________________________________________________________________________________
(15, 3000000)


In [221]:
train_data = []

for i in range(0, 3000000 - 10000, 4000):
    train_data.append((i_data[:, i:i+5000], 0))
    train_data.append((p_data[:, i:i+5000], 1))

In [223]:
random.seed(42)  # Optional: for reproducibility
random.shuffle(train_data)

### Model

In [326]:
class DynamicCorrGCN(nn.Module):
    def __init__(self, in_channels=1, hidden_channels=16, num_classes=2):
        super(DynamicCorrGCN, self).__init__()
        self.gcn1 = GCNConv(in_channels, hidden_channels)
        self.gcn2 = GCNConv(hidden_channels, hidden_channels)
        self.fc = nn.Linear(hidden_channels, num_classes)
        self.dropout = nn.Dropout(p=0.3)

    def compute_edge_index_from_corr(self, segment, threshold=0.5):
        """
        segment: Tensor of shape [n_channels, time_steps]
        Returns: edge_index for GCN
        """
        with torch.no_grad():
            corr = torch.corrcoef(segment)  # [n_channels, n_channels]
            corr[torch.isnan(corr)] = 0.0
            adj = (torch.abs(corr) > threshold).float()
            adj.fill_diagonal_(1)  # add self-loops
            edge_index, _ = dense_to_sparse(adj)
            print("segment shape:", segment.shape)
            corr = torch.corrcoef(segment)
            print("corr shape:", corr.shape)
        return edge_index

    def forward(self, segment):
        """
        segment: [1, n_channels, time_steps] — batch of size 1
        Returns: [1, num_classes] logits
        """
        edge_index = self.compute_edge_index_from_corr(segment)

        # Node features = mean over time → [n_channels, 1] YOU CAN CHANGE THIS, NO ONE USES MEAN because this averages over time
        x = segment.mean(dim=1, keepdim=True)

        x = F.relu(self.gcn1(x, edge_index))
        
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, edge_index))
 

        # Final classification layer
        out = self.fc(x)  # [1, num_classes]
        return out


In [328]:
model = GCN_model(in_channels=1, hidden_channels=16, num_classes=2)
optimizer = optim.Adam(model.parameters(), lr=0.001,weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

In [330]:
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(100):
    total_loss = 0.0
    correct = 0

    for segment, label in train_data:
        optimizer.zero_grad()

        # Convert to tensors
        segment = torch.tensor(segment, dtype=torch.float32).unsqueeze(0)  # [1, n_channels, time_steps]
        label = torch.tensor([label], dtype=torch.long)

        # Forward pass
        output = model(segment, label)  # returns logits [1, 2]
        loss = criterion(output, label)

        # Backward + optimize
        loss.backward()
        optimizer.step()

        # Accuracy
        pred = output.argmax(dim=1)
        correct += (pred == label).item()
        total_loss += loss.item()

    acc = 100 * correct / len(train_data)
    avg_loss = total_loss / len(train_data)
    if epoch%10 ==0:
        print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, Accuracy = {acc:.2f}%")


Epoch 1: Loss = 1.0987, Accuracy = 59.02%
Epoch 11: Loss = 0.5257, Accuracy = 75.53%
Epoch 21: Loss = 0.5092, Accuracy = 76.14%
Epoch 31: Loss = 0.6502, Accuracy = 62.43%
Epoch 41: Loss = 0.5174, Accuracy = 79.41%
Epoch 51: Loss = 0.5412, Accuracy = 77.87%
Epoch 61: Loss = 0.5271, Accuracy = 78.14%
Epoch 71: Loss = 0.4961, Accuracy = 79.55%
Epoch 81: Loss = 0.4943, Accuracy = 79.81%
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Applications/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/3f/pljvlty17172k_x4svc7myr00000gn/T/ipykernel_5222/1428683148.py", line 17, in <module>
    output = model(segment, label)  # returns logits [1, 2]
             ^^^^^^^^^^^^^^^^^^^^^
  File "/Applications/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1532, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Applications/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1541, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/3f/pljvlty17172k_x4svc7myr00000gn/T/ipykernel_5222/2752661124.py", line 38, in forward
    batch = Batch.from_data_list(graphs)
            ^^^^^^^^^^^^^^^^^^^

### testing


In [296]:
interictal_2 = '//Users/poorvaichandrasen/Downloads/Patient_1/Patient_1_interictal_segment_0002.mat'
i_eg_2 = io.loadmat(interictal_2)
interictal_segment_2 = i_eg_2.get('interictal_segment_2')
preictal_2 = '//Users/poorvaichandrasen/Downloads/Patient_1/Patient_1_preictal_segment_0002.mat'
p_eg_2 = io.loadmat(preictal_2)
preictal_segment_2 = p_eg_2.get('preictal_segment_2')

In [298]:
i_t_data= interictal_segment_1[0][0][0]
p_t_data = preictal_segment_1[0][0][0]


In [320]:
test_data = []

for i in range(7000,10000, 200):
    test_data.append((i_data[:, i:i+5000], 0))
    test_data.append((p_data[:, i:i+5000], 1))
random.shuffle(test_data)


In [322]:
model.eval()
correct = 0
total = 0

with torch.no_grad():  # Disable gradient tracking for faster inference
    for i in range(len(test_data)):
        segment = test_data[i][ 0]
        label = test_data[i][ 1]
        segment = torch.tensor(segment, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(label).item()  # convert to Python int

        output = model(segment)  # shape: [1, 2]
        pred = output.argmax(dim=1).item()  # predicted class index (0 or 1)

        correct += (pred == label)
        total += 1
        accuracy = 100 * correct / total
        print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 100.00%
Test Accuracy: 100.00%
Test Accuracy: 66.67%
Test Accuracy: 50.00%
Test Accuracy: 40.00%
Test Accuracy: 50.00%
Test Accuracy: 42.86%
Test Accuracy: 37.50%
Test Accuracy: 44.44%
Test Accuracy: 50.00%
Test Accuracy: 54.55%
Test Accuracy: 58.33%
Test Accuracy: 53.85%
Test Accuracy: 57.14%
Test Accuracy: 60.00%
Test Accuracy: 62.50%
Test Accuracy: 58.82%
Test Accuracy: 55.56%
Test Accuracy: 57.89%
Test Accuracy: 60.00%
Test Accuracy: 57.14%
Test Accuracy: 54.55%
Test Accuracy: 52.17%
Test Accuracy: 50.00%
Test Accuracy: 52.00%
Test Accuracy: 50.00%
Test Accuracy: 51.85%
Test Accuracy: 50.00%
Test Accuracy: 48.28%
Test Accuracy: 50.00%


torch.Size([1, 15, 5000])